## 1. Preparation

Before you start, please **verify that you are using a GPU.**  
In the Menu, select "Runtime" -> "Change runtime type"

Then, we are ready to clone the course repository and upload all required packages.

In [ ]:
!git clone https://github.com/SimoneRebora/CMCLS

import pandas as pd
from sklearn.metrics import classification_report
from datetime import datetime
import io
import os
import re
import numpy as np

from transformers import AutoModelForSequenceClassification, AutoTokenizer, Trainer, TrainingArguments
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.model_selection import KFold

Read the datasets

In [ ]:
# 1. Read the first curated dataset (100 passages)
df_1 = pd.read_csv(
    '/content/CMCLS/materials/HoD_annotated_dataset.csv',
    encoding='utf-8',
    encoding_errors='replace'
)[['Sentences', 'curation']]

# 2. Read the class curated dataset (60 passages) and align column name
df_2 = pd.read_csv(
    '/content/CMCLS/materials/Hands_on_5_curation_first_round.csv',
    encoding='utf-8',
    encoding_errors='replace'
)[['text', 'curation']].rename(columns={'text': 'Sentences'})

# 3. Merge into 160 passages
my_df = pd.concat([df_1, df_2], ignore_index=True)

# 4. Remove empty rows and standardize text/labels
my_df = my_df.dropna(subset=['Sentences', 'curation']).copy()
my_df['Sentences'] = my_df['Sentences'].astype(str).str.strip()
my_df['curation'] = my_df['curation'].astype(str).str.strip().str.upper()

# 5. Rename columns
df = my_df
df = df.rename(columns={
    'Sentences': 'text',
    'curation': 'label'
})

# Show
df

## 2. Finetune a pretrained model

In [ ]:
# define model to use
checkpoint = "google-bert/bert-base-uncased"

# define unique labels
unique_labels = set(label for label in df['label'])

# Define training arguments
training_args = TrainingArguments(
    num_train_epochs=3,              # total number of training epochs
    per_device_train_batch_size=16,  # batch size per device during training
    per_device_eval_batch_size=64,   # batch size for evaluation
    warmup_steps=500,                # number of warmup steps for learning rate scheduler
    weight_decay=0.01,               # strength of weight decay
    logging_steps=10,
    output_dir="./bert_finetuned",
    eval_strategy="epoch" # Evaluate every epoch
)

# Define Dataset function
class MyDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

# Define compute_metrics function
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="binary",
        zero_division=0
    )

    accuracy = accuracy_score(labels, predictions)

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

# Split DataFrame into train and test sets
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    train_size=0.8,
    random_state=123,
    shuffle=True
)

curated_texts = test_df['text'].tolist()
curated_labels = test_df['label'].tolist()

label2id = {label: id for id, label in enumerate(unique_labels)}
id2label = {id: label for label, id in label2id.items()}

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForSequenceClassification.from_pretrained(checkpoint,
                                                           num_labels=len(unique_labels),
                                                           id2label=id2label,
                                                           label2id=label2id,
                                                           ignore_mismatched_sizes=True)

# split dataset into train and validation sets
train_ratio = 0.8

train, val, y_train, y_val = train_test_split(train_df['text'], train_df['label'], test_size=1-train_ratio, random_state=123)

train_labels_encoded = [label2id[y] for y in y_train]
test_labels_encoded  = [label2id[y] for y in y_val]

# Tokenize data
train_encodings = tokenizer(train.tolist(), truncation=True, padding=True)
test_encodings = tokenizer(val.tolist(), truncation=True, padding=True)

train_dataset = MyDataset(train_encodings, train_labels_encoded)
test_dataset = MyDataset(test_encodings, test_labels_encoded)

# Finetuning

# Initialize Trainer
trainer = Trainer(
    model=model,                         # the instantiated 🤗 Transformers model to be trained
    args=training_args,                  # training arguments, defined above
    train_dataset=train_dataset,         # training dataset
    eval_dataset=test_dataset,           # evaluation dataset
    compute_metrics=compute_metrics      # the function to evaluate results
)

# Train the model
trainer.train()

# Save fine-tuned model and tokenizer
trainer.save_model("./bert_finetuned")
tokenizer.save_pretrained("./bert_finetuned")

# make predictions on curated set and report scores
curated_encodings  = tokenizer(curated_texts, truncation=True, padding=True)
curated_labels_encoded  = [label2id[y] for y in curated_labels]
curated_dataset = MyDataset(curated_encodings, curated_labels_encoded)

predicted_results = trainer.predict(curated_dataset)

predicted_labels = predicted_results.predictions.argmax(-1) # Get the highest probability prediction
predicted_labels = predicted_labels.flatten().tolist()      # Flatten the predictions into a 1D list
predicted_labels = [id2label[l] for l in predicted_labels]  # Convert from integers back to strings for readability

report = classification_report(curated_labels, predicted_labels)
print(report)

Explore the classification errors with a confusion matrix

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix, f1_score

cm = confusion_matrix(curated_labels, predicted_labels)
macro_f1 = f1_score(curated_labels, predicted_labels, average="macro")

fig, ax = plt.subplots(figsize=(8, 4))

disp = ConfusionMatrixDisplay(cm, display_labels=["NO", "YES"])
disp.plot(ax=ax, cmap="Blues", colorbar=False, values_format="d")

ax.set_title("Confusion Matrix")

# Text annotations on the right
annotation = (
    f"Model:  {checkpoint}\n"
    f"Macro F1-score:  {macro_f1:.2f}"
)

fig.text(
    0.68, 0.5, annotation,
    fontsize=11,
    verticalalignment="center",
    family="monospace"
)

plt.tight_layout(rect=[0, 0, 0.95, 1])
plt.savefig("confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. Use the model on a new sentence

In [ ]:
# Sentence to annotate
sentence = "He was black and poor and ugly."

from transformers import pipeline

# Load fine-tuned model
classifier = pipeline(
    "text-classification",
    model="./bert_finetuned",
    tokenizer="./bert_finetuned"
)

# Predict
prediction = classifier(sentence)[0]

print("Sentence:", sentence)
print("Predicted label:", prediction["label"])
print("Confidence:", round(prediction["score"], 4))

## Your turn

Try a different model, for example: https://huggingface.co/nlptown/bert-base-multilingual-uncased-sentiment